In [ ]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score, classification_report

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam

from nltk.corpus import stopwords
import nltk

In [ ]:
df = pd.read_csv("fr_dataset.csv", sep=';')


df[['labels', 'text']] = df['labels,text'].str.split(',', n=1, expand=True)
df['text'] = df['text'].str.strip('"')


texts = df["text"]
labels = df["labels"].map({"ham": 0, "spam": 1})

In [ ]:
print(labels)

0       0
1       0
2       1
3       0
4       0
       ..
5559    1
5560    0
5561    0
5562    0
5563    0
Name: labels, Length: 5564, dtype: int64


In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"\d+", "", text)          # supprime les chiffres
    text = re.sub(r"[^\w\s]", "", text)      # supprime les ponctuations
    text = re.sub(r"\s+", " ", text).strip()
    return text



In [ ]:
texts = texts.apply(clean_text)

In [ ]:
nltk.download("stopwords")

french_stopwords = stopwords.words("french")

vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words=french_stopwords,
    lowercase=True,
    ngram_range=(1, 2)
)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
X = vectorizer.fit_transform(texts)
y = labels.values

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [ ]:
input_dim = X_train.shape[1]

model = Sequential()
model.add(Dense(128, activation="relu", input_shape=(input_dim,)))
model.add(Dropout(0.3))
model.add(Dense(64, activation="relu"))
model.add(Dropout(0.3))
model.add(Dense(1, activation="sigmoid"))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=15,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

Epoch 1/15
112/112 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - accuracy: 0.8649 - loss: 0.3497 - val_accuracy: 0.8575 - val_loss: 0.1978
Epoch 2/15
112/112 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.9708 - loss: 0.0921 - val_accuracy: 0.9776 - val_loss: 0.0984
Epoch 3/15
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9938 - loss: 0.0216 - val_accuracy: 0.9787 - val_loss: 0.1087
Epoch 4/15
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9983 - loss: 0.0078 - val_accuracy: 0.9798 - val_loss: 0.1197
Epoch 5/15
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.9994 - loss: 0.0045 - val_accuracy: 0.9809 - val_loss: 0.1293
Epoch 6/15
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9997 - loss: 0.0024 - val_accuracy: 0.9798 - val_loss: 0.1326
Epoch 7/15
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9997 - loss: 0.0022 - val_accuracy: 0.9798 - val_loss: 0.1396
Epoch 8/15
112/112 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.9997 - loss: 0.0025 - val_accu

In [ ]:
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob >= 0.5).astype(int)

print("\nAccuracy :", accuracy_score(y_test, y_pred))
print("F1-score :", f1_score(y_test, y_pred))
print("\nRapport de classification :\n")
print(classification_report(y_test, y_pred))

35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step

Accuracy : 0.9757412398921833
F1-score : 0.9039145907473309

Rapport de classification :

              precision    recall  f1-score   support

           0       0.98      0.99      0.99       967
           1       0.94      0.87      0.90       146

    accuracy                           0.98      1113
   macro avg       0.96      0.93      0.95      1113
weighted avg       0.98      0.98      0.98      1113



In [ ]:
def predict_message(message):
    message = clean_text(message)
    vector = vectorizer.transform([message]).toarray()
    prob = model.predict(vector)[0][0]

    label = "SPAM" if prob >= 0.5 else "HAM"
    prob = 1 - prob if prob <= 0.5 else prob
    return label, prob

In [ ]:
test_message = "gagné clique lien appelle numero"
label, confidence = predict_message(test_message)

print("\nMessage :", test_message)
print("Résultat :", label)
print("Confiance :", round(confidence * 100, 2), "%")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step

Message : gagné clique lien appelle numero
Résultat : SPAM
Confiance : 79.24 %


In [ ]:
import pickle
pickle.dump(vectorizer, open("spam_vectorizer.pkl", "wb"))


In [ ]:
model.save("spam_model.keras")

## Convert Keras Model to ONNX

To convert the saved Keras model (`spam_model.keras`) to ONNX format, we will use the `tf2onnx` library. This library allows you to convert TensorFlow/Keras models into ONNX format, which can then be used with various ONNX runtime environments.

In [ ]:
!pip install tf2onnx

In [ ]:
import tensorflow as tf
import tf2onnx
import onnx


keras_model_path = "spam_model.keras"
keras_model = tf.keras.models.load_model(keras_model_path)


onnx_model_path = "spam_model.onnx"


spec = (tf.TensorSpec(keras_model.inputs[0].shape, keras_model.inputs[0].dtype, name="input"),)

onnx_model_proto, _ = tf2onnx.convert.from_keras(keras_model, input_signature=spec, opset=13, output_path=onnx_model_path)

print(f"Keras model successfully converted to ONNX and saved to {onnx_model_path}")